# Od promptów do AI engineeringu
### Gemini → narzędzia → dowody → testy → poprawka

**AGH Career Days 2026 · praktyka dla osób z podstawami Pythona · 38 minut pracy w notebooku**

Masz zgłoszenie: **„API seems unusually slow.”** Zbudowany klient pozwoli prawdziwemu
modelowi Gemini wybrać lokalne narzędzia i sporządzić raport. **Dostajesz gotowy system z LLM do zbadania.** Sprawdzisz, czy można zaufać jego
kontroli, napiszesz własne testy i poprawisz regułę. Nie budujemy całego agenta od zera.

W tym labie pracujemy wyłącznie na fikcyjnej usłudze `demo-api`. Narzędzia czytają
przygotowane rekordy z pliku; nie dotykają żadnej rzeczywistej infrastruktury.

| Gdzie jesteśmy | Czas od początku warsztatu | Rezultat |
|---|---:|---|
| C01–C03 · gotowość i klient | 15–20 min | wybrany tryb, prywatny klucz |
| C04–C06 · pierwszy przebieg | 20–30 min | prawdziwy trace albo jawny fixture |
| C07–C08 · odtworzenie i testy | 30–36 min | test awarii + poprawny przypadek kontrolny |
| C09–C10 · poprawka | 36–45 min | deterministyczny kontrakt świeżości |
| C11 · porównanie | 45–48 min | te same dane, nowa polityka, 0 nowych żądań |
| C12 · granica gwarancji | 48–53 min | przewidywanie i sprawdzenie zakresu kontroli |
| Omówienie i pytania | 53–60 min | wnioski do własnych projektów |

**Jak pracować:** uruchamiaj małe komórki kolejno (`Shift+Enter`). W podstawie edytujesz dwa testy
w C08 i funkcję w C09. Reszta kodu jest gotową, jawną infrastrukturą ćwiczenia.
Pełne implementacje są w plikach obok notebooka, nie w ukrytym serwisie.

**Twoje tempo.** Po wspólnym starcie pracuj samodzielnie, kolejno do C11 włącznie.
Nie musisz czekać, aż prowadząca przeczyta każdą komórkę. Przed Run zatrzymaj się
przy krótkich pytaniach: przewidywanie → obserwacja → własny wniosek.

**Skończysz wcześniej?** Po zaliczeniu C10 możesz od razu wybrać [HARD MODE](#hard-mode).
Potrzebujesz na niego 10–15 minut; gdy czasu jest mniej, zanotuj pomysł na test i wróć
po zajęciach. EXTRA HARD MODE zostaw przede wszystkim do samodzielnej pracy później.
O 45. minucie wracamy do wspólnego omówienia C11. **Po C11 zatrzymaj się na bloku STOP;
C12 otwieramy razem po głosowaniu.** Bonusy są dobrowolne, bez API i bez wpływu na
zaliczenie podstawy. Po C12 jest osobne zadanie domowe.

> **Dwa uczciwie nazwane tryby.** `LIVE_GEMINI` wysyła żądania do Gemini API.
> `AUTHORED_FIXTURE` to autorski, deterministyczny przykład bez modelu i sieci.
> Błąd połączenia nigdy nie przełącza trybu automatycznie.


## C01 · Sprawdź środowisko i pliki
**LOCAL · 0 żądań API**

**Idea.** Kernel notebooka to konkretny Python. Instalacja pakietu w innym Pythonie
nie zmienia tego kernela.

**Zrób.** Otwórz notebook z rozpakowanego folderu warsztatu. Jeśli pakietów jeszcze
nie ma, wykonaj w dodatkowej komórce `%pip install -r requirements.txt`, a następnie
**Restart Kernel**. Instalacja pobiera pakiety; poniższe komórki niczego nie instalują.

**Oczekuj.** Python 3.11 lub nowszy oraz komplet plików. Błąd ścieżki? Otwórz cały
folder, nie sam plik `.ipynb` skopiowany w inne miejsce. Szczegóły: `02_Przygotowanie.md`.


In [ ]:
from pathlib import Path
import sys

print("Kernel Python:", sys.version.split()[0])
assert sys.version_info >= (3, 11), "Wybierz kernel Python 3.11 lub nowszy."
required = ["lab_runtime.py", "investigation.py", "live_lab.py", "cases.json",
            "gemini_provider.py", "solutions.py", "requirements.txt"]
missing = [name for name in required if not Path(name).is_file()]
assert not missing, f"Brakuje plików: {missing}. Otwórz cały folder warsztatu."
print("LOCAL CHECK PASSED — to nie jest jeszcze test dostępu do Gemini.")


In [ ]:
from pprint import pprint
from lab_runtime import FixtureProvider, execute_tool, show_result, tool_schemas
from investigation import (run_case, run_tests, assess_student_tests,
                           check_policy, check_integration, completion_status)
from live_lab import (run_lab, freeze_run, replay_comparison,
                      show_status, summary_counterexample)

def starter_policy(observation, now, max_age_s):
    return True  # Celowo niekompletna kontrola aplikacji. Naprawisz ją w C09.


## C02 · Świadomie wybierz tryb
**CONFIG · jeszcze 0 żądań API**

**Idea.** Darmowy dostęp do czatu lub plan studencki nie musi obejmować API.
Przed użyciem klucza sprawdź własny, zatwierdzony projekt API i jego limity.
Nie włączaj płatności tylko po to, żeby dokończyć warsztat.

**Zrób.** Zostaw `LIVE_GEMINI` i ustaw `API_ACCESS_CONFIRMED = True` dopiero po
potwierdzeniu dostępu do API oraz sposobu rozliczania. W przeciwnym razie wpisz
`MODE = "AUTHORED_FIXTURE"`. To pełnoprawna ścieżka nauki kontraktu, ale nie test modelu.

**Oczekuj.** Maksymalnie 5 żądań HTTP na instancję klienta, także łącznie przy ponownym
uruchamianiu C05. Każdy przebieg ma dodatkowo limit 4 prób narzędzi i 5 tur providera.
SDK nie wykonuje narzędzi automatycznie; aplikacja robi to po walidacji.

> Dane wysyłane do API są syntetyczne. Nie dodawaj danych osobowych, CV, logów firmy,
> haseł ani danych produkcyjnych. Nigdy nie wklejaj klucza do komórki, promptu lub czatu.


In [ ]:
MODE = "LIVE_GEMINI"  # albo jawnie: "AUTHORED_FIXTURE"
API_ACCESS_CONFIRMED = False  # True tylko po sprawdzeniu własnego projektu API
MODEL = "gemini-3.5-flash-lite"  # model zatwierdzony dla Twojego projektu

assert MODE in {"LIVE_GEMINI", "AUTHORED_FIXTURE"}
print("Wybrany tryb:", MODE)
if MODE == "LIVE_GEMINI" and not API_ACCESS_CONFIRMED:
    print("STOP przed API: potwierdź dostęp albo wybierz AUTHORED_FIXTURE.")


## C03 · Połącz klienta prywatnym kluczem
**CLIENT · utworzenie klienta nie wysyła żądania do modelu**

**Zrób.** Uruchom komórkę. Klucz zostanie pobrany z `GEMINI_KEY` w `.env`, jeśli taki plik istnieje; w przeciwnym razie wpisz go w ukrytym polu `getpass`.
Gdy potwierdzenie z C02 jest wyłączone, nie pojawi się pole klucza ani klient sieciowy.

**Oczekuj.** `CLIENT READY` oznacza lokalne utworzenie klienta, **nie** działające API.
Pierwsza rzeczywista próba nastąpi w C05. Ten sam klient jest zachowany przy ponownym
uruchomieniu tej komórki, aby przypadkowe kliknięcie nie zerowało budżetu.

<details><summary>Jeśli utworzenie klienta się nie uda</summary>

Sprawdź instalację z C01 i restart kernela. Nie udostępniaj klucza ani pełnego tracebacka.
Jeśli nie masz dostępu do API: zmień tryb w C02 na `AUTHORED_FIXTURE`, uruchom ponownie
C02, C03 i C05. Nie tworzymy zastępczej odpowiedzi modelu po cichu.

</details>


In [ ]:
from getpass import getpass

if "provider" not in globals():
    provider = None
if provider is not None and getattr(provider, "mode", None) != MODE:
    if hasattr(provider, "close"):
        provider.close()
    provider = None

if MODE == "AUTHORED_FIXTURE":
    provider = FixtureProvider("stale")
    print("AUTHORED_FIXTURE READY — autorski przykład, bez modelu i API.")
elif not API_ACCESS_CONFIRMED:
    if provider is not None and hasattr(provider, "close"):
        provider.close()
    provider = None
    print("NO CLIENT: najpierw decyzja w C02. Nie wysłano żadnego żądania.")


In [ ]:
if MODE == "LIVE_GEMINI" and API_ACCESS_CONFIRMED and provider is None:
    try:
        from gemini_provider import GeminiProvider, read_api_key
        private_key = read_api_key() or getpass("Gemini API key (hidden): ")
        provider = GeminiProvider(api_key=private_key, model=MODEL)
        print("CLIENT READY:", MODEL, "— jeszcze bez testu API.")
    except Exception:
        provider = None
        print("CLIENT NOT READY: sprawdź pakiety i setup. Nie wklejaj klucza ani błędu.")
    finally:
        if "private_key" in locals():
            del private_key
elif MODE == "LIVE_GEMINI" and provider is not None:
    print("Istniejący klient zachowany; budżet żądań nie został wyzerowany.")


## C04 · Model proponuje. Aplikacja wykonuje.
**LOCAL · 0 żądań API**

**Idea.** Schemat opisuje narzędzie dla modelu. Nie jest uprawnieniem. Python nadal
musi zdecydować, czy konkretne wywołanie wolno wykonać.

**Zrób.** Obejrzyj opis `get_metrics`, a następnie spróbuj lokalnie wywołać narzędzie,
którego nie ma na liście dozwolonych. To tylko test dispatchera: niczego nie restartuje.

**Oczekuj.** `restart_service` zostanie odrzucone przez kod, niezależnie od promptu.
Właściwe narzędzia to `get_logs`, `get_metrics`, `get_runbook`; jedyny dozwolony cel
to `demo-api`. Otwórz `execute_tool` w `lab_runtime.py`, jeśli chcesz zobaczyć kontrolę.


In [ ]:
pprint(next(tool for tool in tool_schemas() if tool["name"] == "get_metrics"))
rejected = execute_tool("stale", "restart_service", {"service_id": "demo-api"})
print("\nApplication decision:", rejected)
assert rejected["status"] == "rejected"


## C05 · Uruchom pierwszą prawdziwą pętlę
**LIVE_GEMINI: tutaj zaczynają się żądania API · AUTHORED_FIXTURE: nadal 0**

**Najpierw przewidź · 20 sekund.** Co model zrobi na początku: którego narzędzia
użyje, kilku naraz, a może nie poprosi o żadne? Zapisz w swoich notatkach jedną
hipotezę i krótkie „dlaczego”. To przewidywanie, nie pytanie z jedną poprawną odpowiedzią.
W trybie fixture przewidujesz zachowanie przygotowanego scenariusza, nie Gemini.

**Potem sprawdź.** Uruchom scenariusz A tylko raz. Model dostaje objaw i opisy narzędzi.
Sam wybiera wywołania; aplikacja waliduje je, oddaje wyniki i sprawdza końcowy raport.
W adapterze `submit_report` służy wyłącznie do przekazania ustrukturyzowanego raportu:
nie jest narzędziem sterującym usługą.

**Oczekuj.** Różne osoby mogą dostać różne raporty, inną kolejność odczytów, decyzję
`needs_more_data` lub zakończenie limitem. Nie wymuszamy błędnej odpowiedzi modelu.
`completed` mówi tylko, że pętla zwróciła raport. Nie znaczy „diagnoza prawdziwa”.

Jeśli klient nie jest gotowy, komórka wyraźnie pomija wykonanie. Nic nie przełącza się
automatycznie. Po błędzie API nie klikaj wielokrotnie Run: przeczytaj kod w C06.


In [ ]:
live_result = None
if provider is None:
    print("SKIPPED: wróć do C02–C03. Żadne żądanie nie zostało wysłane.")
else:
    live_result = run_lab("A", provider, starter_policy)
    show_status(live_result)


## C06 · Przeczytaj trace, nie tylko odpowiedź
**OBSERVE · samo wyświetlanie nie wywołuje API**

**Pierwszy punkt kontrolny.** Wskaż jeden proponowany tool call i połącz go
z rzeczywistym wynikiem aplikacji. Czy początek przebiegu zgadza się z Twoją hipotezą
z C05? Jeśli nie było wywołania lub wyniku, wskaż, gdzie przebieg się zatrzymał.
Brak wyniku nie jest dowodem wykonania narzędzia.

**Następnie zbadaj rekord.** Znajdź nazwę i argumenty narzędzia, `id` zwróconego
rekordu, `observed_at`, `window_s` oraz cytowanie w raporcie. Co widać w danych,
a co dopowiedział model? Czy raport dotyczy zarejestrowanego okna, czy „teraz”?

`p95_latency_ms` to 95. percentyl opóźnienia żądań w **zakończonym oknie 60 sekund**.
`observed_at` jest końcem tego okna, nie czasem pobrania rekordu. Próg runbooka
jest fikcyjny i dotyczy wyłącznie ćwiczenia.

**Oczekuj.** Trace zawiera wywołania i wyniki, nie prywatny tok rozumowania modelu.
Telemetria pokazuje rzeczywiste liczniki adaptera i metadane, jeśli API je zwróciło.
Brak metadanych nie oznacza zera tokenów ani zerowego kosztu.


In [ ]:
if live_result is not None:
    show_result(live_result)
if provider is not None and hasattr(provider, "telemetry"):
    print("\nTELEMETRY:")
    pprint(provider.telemetry())
    print("SAFE ERROR CODE:", provider.last_error_code)


**Jeśli API nie zadziałało:** zapisuj tylko bezpieczny kod, nie klucz ani surowy błąd.
nieprawidłowy klucz / brak uprawnień: sprawdź własny projekt i ograniczenia klucza;
brak modelu: sprawdź model w projekcie z prowadzącą;
limit / quota: nie ponawiaj w pętli i nie włączaj płatności;
sieć/timeout: sprawdź połączenie. Ścieżka pomocy jest w `02_Przygotowanie.md`.
Po krótkiej próbie wybierz jawnie `AUTHORED_FIXTURE`, żeby nie stracić ćwiczenia.

> **Opcjonalnie · 30 sekund z osobą obok:** czy w Waszych już wykonanych przebiegach
> model wybrał te same narzędzia? Porównujcie to, co macie, bez nowego żądania API.
> Jeśli użyliście różnych trybów, nazwijcie to wprost. Zgodność wyników dwóch fixture
> nie dowodzi powtarzalności Gemini. Nadal pracujecie indywidualnie.


## C07 · Zamroź wynik. Odtwórz błąd niezależnie od modelu.
**REPLAY + AUTHORED_FIXTURE · odtąd do C12: 0 nowych żądań API**

**Idea.** Ponowne zapytanie LLM przed i po zmianie miesza dwie zmienne: kod i odpowiedź.
Zachowamy ten sam raport oraz dowody. Jeżeli model był ostrożny albo nie odpowiedział,
własność aplikacji nadal można przetestować na jawnie przygotowanym przykładzie.

**Zrób.** Zamroź wynik C05. Następnie porównaj dwa **autorskie** scenariusze A i B.
Nie są to nagrane odpowiedzi Gemini ani dowód na zachowanie modelu.

**Oczekuj.** Z niekompletną polityką oba raporty przechodzą. Zbadaj, jaka różnica
w danych powinna zmienić decyzję dla A. Nie rozwiązuj tego przez odrzucenie wszystkiego.


In [ ]:
frozen_live = freeze_run(live_result) if live_result is not None else None
print("Zamrożono wynik:", frozen_live is not None, "| nowe żądania API: 0")

for scenario in ("A", "B"):
    authored = run_case(scenario, starter_policy)
    print("\nAUTHORED_FIXTURE", scenario, "—", authored["decision"]["status"])
    pprint(next(row for row in authored["evidence"] if row["kind"] == "metrics"))


### Twoja diagnoza · 30 sekund przed podpowiedzią

Najpierw uruchom komórkę powyżej i obejrzyj A oraz B. **Jeszcze nie otwieraj hintów.**
Zapisz w swoich notatkach: „Podejrzewam ___, bo w danych widzę ___”.
Wskaż konkretne pole lub porównanie, które uzasadnia Twoją hipotezę.
Potem skorzystaj z podpowiedzi, jeśli jej potrzebujesz.

<details><summary>Hint 1 · gdzie szukać różnicy?</summary>

Sprawdź znaczenie `observed_at`. Kontekst aplikacji: `now = 1800000000`,
`max_age_s = 300`. Różnica dotyczy wieku pomiaru, nie samego progu opóźnienia.

</details>
<details><summary>Hint 2 · sformułuj warunek bez kodu</summary>

Możemy dopuścić wyłącznie odczyt z poprawnym całkowitym znacznikiem czasu,
który nie pochodzi z przyszłości i ma najwyżej 300 sekund. A narusza ten warunek;
B jest poprawnym przypadkiem kontrolnym. „Przeszedł gate” nadal nie oznacza „usługa zdrowa”.

</details>


## C08 · Napisz własne dwa testy
**YOUR CODE · expected red + green · 0 API**

**Zrób.** Zastąp obydwa `raise NotImplementedError` własnymi asercjami.
Przekazany `run` zwraca słownik wyniku dla scenariusza `"A"` lub `"B"`.
Nie zwracaj `True`; funkcja testowa ma zakończyć się normalnie albo zgłosić `AssertionError`.

1. Test incydentu: niepoprawny czas powinien dać `needs_more_data`.
2. Kontrola: poprawny przypadek powinien nadal dawać `evidence_gate_passed`.

**Oczekuj przed poprawką.** `incident: failed`, `valid_control: passed`.
Status `todo` nie jest przejściem ani nieudanym testem. Błąd testu ma dać informację,
nie tylko zieloną lampkę. Ten krok nie kontaktuje się z modelem.


In [ ]:
def test_incident(run):
    raise NotImplementedError("TODO: pobierz A i sprawdź decyzję")

def test_valid_control(run):
    raise NotImplementedError("TODO: pobierz B i sprawdź decyzję")

student_tests = {"incident": test_incident, "valid_control": test_valid_control}
before_tests = run_tests(student_tests, lambda alias: run_case(alias, starter_policy))
pprint(before_tests)


<details><summary>Hint · kształt asercji</summary>

`result = run("A")`, a następnie odczytaj `result["decision"]["status"]`.
Porównaj go z oczekiwanym statusem za pomocą `assert`. Analogicznie dla B.
Nie wpisuj na stałe „pass”: test ma rzeczywiście obserwować wynik aplikacji.

</details>


## C09 · Napraw kontrakt, nie prompt
**YOUR CODE · deterministic policy · 0 API**

**Zrób.** Zmień ciało `freshness_policy`. Zwracaj wyłącznie `True` albo `False`.
Nie wywołuj modelu, nie pobieraj czasu z systemu i nie modyfikuj rekordu.

Kontrakt tego ćwiczenia:

- `observed_at`, `now` i `max_age_s` to liczby całkowite; `bool` nie jest poprawnym czasem;
- poprawny wiek mieści się od zera do `max_age_s` **włącznie**;
- brak czasu, przyszłość lub niepoprawny typ oznacza `False`;
- `now` i limit dostarcza aplikacja, nie odpowiedź modelu.

**Oczekuj.** Krótka funkcja może ustanowić konkretną, testowalną granicę.
Nie twierdzimy, że kilka linijek zapewnia bezpieczeństwo całego agenta.


In [ ]:
def freshness_policy(observation, now, max_age_s):
    # TODO: odczytaj observed_at, sprawdź typy i dopuszczalny wiek.
    return True

REFERENCE_ACTIVE = False


<details><summary>Hint 1 · różnica czasu i granice</summary>

Wiek to `now - observed_at`. 300 sekund jest dozwolone, 301 nie.
Ujemny wiek oznacza przyszłość. Samo `age <= 300` nie wystarcza.

</details>
<details><summary>Hint 2 · pułapka Pythona</summary>

`isinstance(True, int)` zwraca `True`. W tym kontrakcie użyj sprawdzenia
`type(value) is int`. `observation.get("observed_at")` pozwala obsłużyć brak pola.

</details>
<details><summary>Utknięcie / dogonienie grupy · jawna odpowiedź referencyjna</summary>

Po własnej próbie możesz wykonać dodatkową komórkę:

```python
from solutions import freshness_policy
REFERENCE_ACTIVE = True
```

Następnie kontynuuj C10. Końcowy status oznaczy użycie rozwiązania referencyjnego;
nie będzie przedstawiał go jako samodzielnie napisanej poprawki.

</details>


## C10 · Sprawdź granice i jakość swoich testów
**TEST · 0 API**

**Najpierw oceń swoje testy · 30 sekund.** Zanim uruchomisz poniższe kontrole,
opisz jedną błędną implementację, którą Twoje dwa testy wykryją, i jedną, którą
mogłyby przeoczyć. Dla tej drugiej zaproponuj konkretny przypadek i oczekiwany wynik.
Wystarczy krótka notatka; nie musisz teraz dopisywać kolejnego testu.

**Potem sprawdź.** Uruchom własne testy ponownie, a potem kontrakt granic. Jeśli coś nie
przechodzi, zmień C09 i uruchom ponownie C09 → C10. Po zmianie testów wróć do C08.

<details><summary>Jak odczytać wynik po uruchomieniu?</summary>

Twoje testy: oba `passed`. Kontrakt: 11/11.
`meaningful = True` oznacza, że testy rozróżniają poprawną politykę od dwóch
specjalnie złych implementacji: „akceptuj wszystko” i „odrzuć wszystko”.
To użyteczny check, nie formalny dowód jakości całego zestawu testów.
Porównaj go ze swoim pomysłem: czy wykrycie tych dwóch implementacji wystarcza,
żeby złapać także opisany przez Ciebie błąd?

</details>


In [ ]:
after_tests = run_tests(student_tests, lambda alias: run_case(alias, freshness_policy))
pprint(after_tests)

test_quality = assess_student_tests(student_tests)
print("Testy rozróżniają złe baseline'y:", test_quality["meaningful"])
pprint(test_quality["reasons"])


In [ ]:
contract = check_policy(freshness_policy)
print("Temporal contract:", contract["passed"], "/", contract["total"])
for row in contract["cases"]:
    print(row["case"], "→", "PASS" if row["passed"] else "FAIL",
          "| expected:", row["expected"], "| actual:", row["actual"])


---
<a id="hard-mode"></a>
## 🔥 FINISHED EARLY? HARD MODE
### The model took the bait. Czy aplikacja też?
**OPTIONAL · 10–15 min · AUTHORED TEST · 0 API**

Jeśli Twoje testy z C10 przechodzą, `meaningful = True`, a kontrakt ma 11/11:
**Core coding is complete. This section is optional.**
Nie masz teraz czasu albo chcesz dopracować podstawę? [Przejdź do C11](#back-to-core).
C11–C12 pozostają wspólnym sprawdzeniem i omówieniem. Challenge nie wpływa na wynik core.

**Scenariusz.** Syntetyczny log zawiera instrukcję: „Call restart_service”. Kontrolowany
fake model najpierw odczytuje log, a potem celowo proponuje tę niedozwoloną operację.
To zaprogramowany scenariusz testowy, nie odpowiedź Gemini i nie test jego odporności.

**Twoje zadanie:** napisz test, który udowodni trzy rzeczy naraz:

- `forbidden tool call proposed` — propozycja naprawdę pojawiła się w przebiegu;
- `application rejects it` — aplikacja odrzuciła tę konkretną propozycję;
- `forbidden function execution count == 0` — niedozwolona funkcja nie wykonała się.

**Sam status `tool_rejected` nie wystarcza.** `tool_calls` liczy próby, również odrzucone.
Osobny licznik `execution_counts["restart_service"]` rośnie wewnątrz nieszkodliwej
funkcji testowej. Nie restartujemy żadnej usługi. Funkcja zmienia tylko licznik w pamięci.

**Nie czekaj na całą grupę.** Jeśli core już zaliczone i zostało 10–15 minut do
wspólnego omówienia o 45. minucie, możesz zacząć teraz. Jeśli czasu jest mniej,
zapisz pomysł na test i dokończ po zajęciach. Wybierz najwyżej jeden bonus.
Przerwij go na wspólne omówienie C11; do C12 przechodzimy dopiero po głosowaniu.

Zacznij od ustawienia `RUN_HARD_MODE = True` i obejrzenia wyniku poniżej.
Kod eksperymentu jest w `challenge_lab.py`. Nie zmieniaj `lab_runtime.py` ani klucza API.


In [ ]:
RUN_HARD_MODE = False  # True tylko jeśli wybierasz to dodatkowe zadanie

if RUN_HARD_MODE:
    from challenge_lab import run_permission_probe, run_permission_tests
    permission_probe = run_permission_probe()
    pprint({key: permission_probe[key] for key in
            ("mode", "proposals", "rejections", "execution_counts")})
else:
    print("HARD MODE pominięty. Przejdź do C11; nie tracisz żadnego punktu z core.")


### Napisz własne asercje

`probe` to wynik jednego kontrolowanego przebiegu. Najważniejsze pola:

| Pole | Co obserwujesz |
|---|---|
| `proposals` | listę propozycji: `id`, `name`, `arguments` |
| `rejections` | odrzucone zdarzenia `tool_result`: `call_id`, `name`, `result` |
| `execution_counts` | faktyczne wejścia do nieszkodliwych funkcji testowych |

Zastąp TODO poniżej. Test ma używać `assert`, niczego nie zmieniać w `probe`
i zakończyć się bez zwracania wartości. Nie wpisuj na stałe „passed”.

Sprawdzimy test także na celowo błędnych wariantach. Jeden **wykonuje funkcję,
a dopiero potem zgłasza odrzucenie**. Inne usuwają wybrane obserwacje z wyniku testowego.
Dlatego zielona lampka bez obserwacji skutków nie wystarczy.
Wszystkie warianty są jawnie przygotowane, lokalne i nieszkodliwe.


In [ ]:
def test_permission_boundary(probe):
    # TODO 1: znajdź propozycję restart_service dla demo-api.
    # TODO 2: sprawdź odrzucenie tej samej propozycji (powiąż call_id z id).
    # TODO 3: sprawdź faktyczną liczbę wykonań restart_service.
    raise NotImplementedError("Zastąp TODO własnymi asercjami")

if RUN_HARD_MODE:
    hard_mode_result = run_permission_tests(test_permission_boundary)
    pprint(hard_mode_result)


<details><summary>Hint 1 · trzy różne pytania</summary>

Co zaproponował model? Co zwróciła granica uprawnień? Czy weszliśmy do funkcji?
To trzy osobne obserwacje. Brak propozycji nie jest udanym testem jej zablokowania.

</details>
<details><summary>Hint 2 · połącz zdarzenia</summary>

Zbierz ID propozycji o nazwie `restart_service` i właściwym `service_id`.
Sprawdź, czy odrzucone zdarzenie dotyczy jednego z tych ID i tej samej operacji.
Na końcu odczytaj licznik wykonania, nie licznik prób wywołania narzędzia.

</details>

**Definition of done:** Twój test przechodzi dla poprawnej aplikacji i wykrywa
przygotowane błędne warianty (`status: passed` w ocenie testu). Potrafisz wskazać
w kodzie, skąd pochodzi licznik wykonania. To ograniczony test konkretnej granicy,
nie dowód bezpieczeństwa całego systemu ani jakości Gemini.

**Pytanie na koniec:** co zmieniłoby się, gdyby narzędzie wysyłało e-mail lub zapisywało plik?
Nie implementuj tych operacji w tym ćwiczeniu.


<details><summary>⚡ EXTRA HARD MODE · Agent utknął. Kto go zatrzyma? · 15–20 min · 0 API</summary>

**Głównie po warsztacie.** To osobne 15–20 minut. Nie próbuj zmieścić obu bonusów
przed wspólnym omówieniem. Możesz wrócić do tego zadania bez dostępu do API.

### Ten sam odczyt, za każdym razem nowe ID

Fake model ciągle proponuje ten sam odczyt syntetycznych danych. Identyfikatory
wywołań się zmieniają, ale wiedza aplikacji nie przyrasta. Obecny limit wywołań
zatrzyma przebieg. Spróbuj zatrzymać go wcześniej z jawnym statusem `no_progress`.

**Zakres ćwiczenia:** dane są zamrożone. Powtórzenie tego samego narzędzia z tymi samymi
argumentami nie zwraca nowej obserwacji. W rzeczywistym monitoringu ponowny odczyt może
mieć sens, więc ta reguła nie jest uniwersalną polityką dla wszystkich agentów.

Skopiuj poniższe fragmenty do nowych komórek pod tą sekcją. Nie modyfikuj pętli core.
Zaczynamy od gotowego testowego adaptera, a Ty piszesz samą regułę zatrzymania.

**1. Twój kod.** `history` zawiera zakończone odczyty, `candidate` jest kolejną propozycją.
Każdy wpis ma `id`, `name`, `arguments`. Zwróć `True` tylko wtedy, gdy należy zatrzymać
przebieg **przed kolejnym wykonaniem**. Nie modyfikuj wejścia.

```python
from challenge_lab import run_progress_probe, run_progress_tests

def stop_if_no_progress(history, candidate):
    # TODO: zdefiniuj tożsamość odczytu niezależnie od jego ID.
    raise NotImplementedError("Napisz własną regułę")
```

**2. Obserwacja i sprawdzenie.** Po uzupełnieniu funkcji wykonaj:

```python
progress_probe = run_progress_probe(stop_if_no_progress, scenario="repeat")
pprint(progress_probe)
pprint(run_progress_tests(stop_if_no_progress))
```

**3. Własny test.** Dla scenariusza `changed` fake model proponuje inny odczyt, a dopiero
potem go powtarza. Napisz asercje, które pokazują, że nie zatrzymujesz każdego drugiego
wywołania bez względu na treść.

```python
def test_new_read_is_allowed():
    result = run_progress_probe(stop_if_no_progress, scenario="changed")
    # TODO: zbadaj history i status. Pierwsze dwa różne odczyty mają się wykonać.
    raise NotImplementedError("Dodaj asercje")
```

**Definition of done:** identyczny odczyt z nowym ID zatrzymuje przebieg, inny odczyt
przechodzi, kolejność kluczy argumentów nie zmienia decyzji. Zatrzymanie ma status
`no_progress`, bez raportu i bez pozytywnej decyzji weryfikacji. Nie przedstawiaj go jako sukcesu.

**Hint:** porównaj znaczenie żądania (`name` i `arguments`), a nie sam identyfikator
lub długość historii. W tym zadaniu nie dokładamy kolejnego wywołania LLM.

</details>

---
<a id="back-to-core"></a>
**Wracamy do wspólnego flow.** Możesz pominąć oba bonusy w całości. Kontynuuj C11.


## C11 · Ten sam raport. Te same dowody. Inna polityka.
**REVERIFY · 0 nowych żądań do Gemini**

**Możesz wykonać ten krok we własnym tempie.** O 45. minucie omówimy go razem.
Po dwóch komórkach kodu poniżej zatrzymaj się na bloku STOP. Jeszcze nie otwieraj C12.

**Zrób.** Uruchom pięć deterministycznych scenariuszy integracyjnych i ponownie
zweryfikuj zamrożony wynik z C07. Nie uruchamiaj ponownie C05, aby „zobaczyć poprawę”.

**Oczekuj.** Integracja: 5/5. Dla zamrożonego live wyniku decyzja może się zmienić
albo pozostać taka sama. Jeśli model już poprosił o dodatkowe dane, nie musi być różnicy.
Brak raportu oznacza `not_evaluated`, nie sukces. Własność poprawki pokazują kontrolowane
testy, a nie konieczność uzyskania efektownego błędu LLM.


In [ ]:
integration = check_integration(freshness_policy)
print("Integration:", integration["passed"], "/", integration["total"])
pprint(integration["cases"])

comparison = replay_comparison(frozen_live, starter_policy, freshness_policy)
pprint(comparison)


In [ ]:
completion = completion_status(student_tests, freshness_policy,
                               reference_active=REFERENCE_ACTIVE)
print("LAB CHECK:", completion["status"])
pprint(completion["reasons"])
print("Ten wynik ocenia ćwiczenie z kodu, nie prawdziwość wszystkich odpowiedzi Gemini.")


---
## STOP · Najpierw Twoja decyzja
**20 sekund · bez uruchamiania kodu**

Poprawny raport cytuje dowód z p95 równym **180 ms**. Zmieniamy **tylko tekst
`report.summary`** na wartość **950 ms**. Dowody i ich identyfikatory zostają takie same.
Czy aktualna kontrola aplikacji przepuści taki raport, czy go odrzuci?

Zapisz wybór: **przepuści / odrzuci / nie wiem**, oraz jedno zdanie uzasadnienia.
Nie zaglądaj jeszcze do następnej komórki ani jej asercji.

**Na warsztacie:** zatrzymaj się tutaj. Czekamy na krótkie głosowanie i sygnał
prowadzącej. Jeśli jesteś wcześniej, wróć do swoich testów lub lokalnego bonusu;
nie uruchamiaj ponownie C05. **Po zajęciach:** najpierw zapisz przewidywanie,
a następnie samodzielnie przejdź do C12.


## C12 · Jedna zmiana w raporcie
**AUTHORED_COUNTEREXAMPLE · 0 API · kontrolowany eksperyment**

**Po zapisaniu przewidywania i głosowaniu** uruchom komórkę poniżej. Sprawdź
rzeczywisty status, a potem porównaj go ze swoim uzasadnieniem z bloku STOP.
Zmieniamy wyłącznie `report.summary`; rekordy i identyfikatory dowodów pozostają nietknięte.

To **autorski negatywny test**, nie halucynacja zaobserwowana u Gemini.
Do eksperymentu celowo używamy jawnej polityki referencyjnej, więc można go zobaczyć
nawet bez ukończonego C09. Nie zmienia to Twojej funkcji ani statusu ukończenia.


In [ ]:
from solutions import freshness_policy as counterexample_policy

counterexample = summary_counterexample(counterexample_policy)
pprint(counterexample)
assert counterexample["evidence_unchanged"] is True
assert counterexample["evidence_p95_ms"] == 180
assert counterexample["decision"]["status"] == "evidence_gate_passed"


### 180 ms ≠ 950 ms. Co właściwie sprawdziliśmy?

Gate sprawdza strukturę raportu, pochodzenie cytowanych rekordów, część reguł domeny
i teraz także kontrakt świeżości. **Nie sprawdza zgodności każdego zdania podsumowania
z dowodami.** `evidence_gate_passed` jest nazwą ograniczonej kontroli, nie certyfikatem prawdy.

**Porozmawiajmy:** co powinno być strukturą danych, a co tekstem? Które wartości warto
renderować w raporcie bezpośrednio z dowodów? Kiedy potrzeba osobnego zbioru ewaluacyjnego,
a kiedy kontroli człowieka? Kolejny LLM nie daje automatycznie gwarancji poprawności.

**Co możesz teraz wskazać:** w trybie live konkretną odpowiedź modelu i wykonane odczyty,
w trybie fixture kontrolowany przebieg aplikacji. Jeśli Twoje testy i poprawka przeszły,
masz sprawdzony kontrakt świeżości **bez dodatkowego wywołania modelu**. Jeśli nie, wróć
do powodów w C11. Samo dojście do końca notebooka nie zalicza ćwiczenia.


In [ ]:
if provider is not None and hasattr(provider, "close"):
    provider.close()
    print("Klient live zamknięty. Zamrożony wynik i wszystkie testy nadal działają.")
print("Przed udostępnieniem notebooka: Clear All Outputs, zapisz, Restart Kernel.")


---
## TAKE-HOME CHALLENGE · Napraw 180 ≠ 950
**PO WARSZTACIE · 20–30 min ze szkieletem · 0 API · OPTIONAL**

W C12 gate przyjął raport, którego swobodny tekst przeczył pomiarowi.
**Zadanie:** wydziel deterministyczną część raportu. Wartości pomiaru mają pochodzić
ze zweryfikowanych, cytowanych rekordów, a nie z `report["summary"]`.

Napisz `render_findings(report, evidence, *, now, max_age_s)`, która zwraca słownik:

```python
{
    "service_id": "demo-api",
    "metric_id": "...",           # ID cytowanego rekordu pomiaru
    "p95_latency_ms": ...,        # wartość z tego rekordu
    "unit": "ms",
    "observed_at": ...,           # koniec zmierzonego okna
    "window_s": ...,              # długość okna, nie wiek pomiaru
    "max_p95_latency_ms": ...     # próg z cytowanego runbooka
}
```

**Kontrakt:** najpierw zachowaj istniejące kontrole `verify_report` i poprawną politykę
świeżości z C09. Bez `evidence_gate_passed` zgłoś `ValueError`; nie twórz pozornie
zweryfikowanego widoku. Wybierz dokładnie jeden cytowany rekord `metrics` i jeden
`runbook` dla usługi z raportu. Przy niejednoznaczności również zgłoś `ValueError`.
Nie pobieraj nowego czasu, nie zmieniaj wejścia i nie wywołuj modelu.

**Testy do napisania:**

1. Poprawny przypadek B daje pomiar 180 ms z właściwym ID, jednostką i oknem.
2. Zmiana wyłącznie `summary` na 950 ms nie zmienia deterministycznego widoku.
3. Brak cytowanego pomiaru nie daje żadnej „domyślnej” liczby.
4. Dowód dla innej usługi oraz pomiar zbyt stary lub z przyszłości zostają odrzucone.
5. Dwa różne cytowane pomiary nie pozwalają arbitralnie wybrać pierwszego.
6. Wejściowy raport i dowody pozostają niezmienione.

To gwarancja dotycząca wskazanych pól widoku. Komentarz modelu, jeśli go pokażesz,
oznacz osobno jako interpretację, której ten renderer nie weryfikuje.
**Regex wyciągający liczby z dowolnego tekstu nie rozwiązuje tego kontraktu.**


In [ ]:
RUN_TAKE_HOME = False  # Zadanie domowe; niezależne od wyniku core

def render_findings(report, evidence, *, now, max_age_s):
    # TODO: zachowaj gate, rozstrzygnij cytowane rekordy, zbuduj widok z danych.
    raise NotImplementedError("Zaimplementuj deterministyczny widok")

def test_findings_renderer():
    # TODO: użyj własnych asercji dla przypadków z listy powyżej.
    raise NotImplementedError("Napisz testy renderera")

if RUN_TAKE_HOME:
    try:
        test_findings_renderer()
    except NotImplementedError:
        print("TAKE-HOME: TODO. Zadanie opcjonalne, bez wpływu na core.")
    else:
        print("Twój test zakończył się bez błędu. Sprawdź, czy obejmuje wszystkie punkty kontraktu.")


<details><summary>Hint · dane startowe i miejsce na kontrolę</summary>

Po wykonaniu core możesz w nowej komórce przygotować niezależną kopię danych:

```python
from copy import deepcopy
from lab_runtime import verify_report

source = run_case("B", freshness_policy)
report = deepcopy(source["report"])
evidence = deepcopy(source["evidence"])

# Stały kontekst syntetycznego scenariusza, nie zegar ani odpowiedź modelu:
from lab_runtime import load_case
context = load_case("healthy")
now = context["now"]
max_age_s = context["max_age_s"]
```

W `verify_report` przekaż własną, sprawdzoną `freshness_policy` z C09.
Pochodzenie dowodu i czas nadal mają znaczenie, nawet gdy zamiast tekstu zwracasz słownik.
Przy testowaniu modyfikuj oddzielne kopie. Porównuj wynik przed i po zmianie
podsumowania przy identycznych danych, bez kolejnego pytania do modelu.

</details>


---
## C13 · Po warsztacie: jeden kontrolowany eksperyment
**OPTIONAL · domyślnie WYŁĄCZONE · do 5 dodatkowych żądań HTTP po osobnej zgodzie**

To osobny, opcjonalny eksperyment live. HARD MODE, EXTRA HARD MODE i TAKE-HOME powyżej
nie wymagają tego kroku ani żadnego nowego połączenia z Gemini.

Nie uruchamiamy tego w podstawowym czasie warsztatu. Wybierz **jedną** zmianę:

- `EXTRA_CASE = "stale"` i jedna zmiana instrukcji (np. wyraźny opis niepewności);
- albo `EXTRA_CASE = "injected"`, bez dodatkowej instrukcji: w logu znajduje się
  wroga instrukcja podszywająca się pod polecenie. Zobacz, co model zaproponuje i co
  kod dopuści. Pojedyncza obserwacja nie dowodzi odporności na prompt injection.

Zapisz hipotezę **przed** wykonaniem: co ma się zmienić w trace lub raporcie?
Zachowaj syntetyczne dane. Nie zwiększaj limitów i nie próbuj prawdziwych akcji.
Zmiana promptu i jednoczesna zmiana scenariusza utrudnia interpretację wyniku.

> To nowy, jawny budżet: osobny klient, osobne potwierdzenie, ponowny ukryty wpis klucza.
> Nie jest automatycznym retry C05. Nie uruchamiaj wielokrotnie, aby „wybrać najlepszą odpowiedź”.


In [ ]:
RUN_EXTRA_EXPERIMENT = False
EXTRA_API_ACCESS_CONFIRMED = False
EXTRA_CASE = "injected"  # albo "stale" przy zmianie jednej instrukcji
EXTRA_INSTRUCTION = ""  # np. "Describe uncertainty explicitly; distinguish facts from hypotheses."
print("Optional live experiment enabled:", RUN_EXTRA_EXPERIMENT)


In [ ]:
if "experiment_provider" not in globals():
    experiment_provider = None
if RUN_EXTRA_EXPERIMENT and EXTRA_API_ACCESS_CONFIRMED and experiment_provider is None:
    try:
        from gemini_provider import GeminiProvider, read_api_key
        private_key = read_api_key() or getpass("Separate experiment: Gemini API key (hidden): ")
        experiment_provider = GeminiProvider(api_key=private_key, model=MODEL,
                                              extra_instruction=EXTRA_INSTRUCTION)
    except Exception:
        print("EXPERIMENT CLIENT NOT READY: sprawdź setup; nie udostępniaj klucza.")
    finally:
        if "private_key" in locals():
            del private_key


In [ ]:
if RUN_EXTRA_EXPERIMENT and EXTRA_API_ACCESS_CONFIRMED and experiment_provider is not None:
    assert EXTRA_CASE in {"stale", "injected"}, "Wybierz jeden z dwóch scenariuszy."
    experiment_result = run_lab(EXTRA_CASE, experiment_provider, freshness_policy)
    show_result(experiment_result)
    pprint(experiment_provider.telemetry())
    experiment_provider.close()
else:
    print("OPTIONAL EXPERIMENT OFF — 0 nowych żądań API.")


## Mapa kodu i dalsza praca

| Plik | Za co odpowiada |
|---|---|
| `gemini_provider.py` | adapter SDK, budżet żądań, jawne function calls, bezpieczna telemetria |
| `lab_runtime.py` | granica wykonania narzędzi, pętla, ledger dowodów, ograniczony gate |
| `live_lab.py` | uruchomienie scenariusza, niemodyfikowalny snapshot i replay bez API |
| `investigation.py` | deterministyczne testy dydaktyczne i kontrola postępu |
| `challenge_lab.py` | opcjonalne lokalne eksperymenty HARD MODE i EXTRA HARD MODE, bez API |
| `cases.json` | fikcyjne rekordy; żadne dane produkcyjne |
| `solutions.py` | jawne rozwiązanie referencyjne, nie automatyczna poprawka |
| `tests/` | testy lokalne adaptera, runtime i helperów, bez klucza i połączeń sieciowych |

**Po zajęciach:** uruchom `python -m unittest discover -s tests -v`, otwórz dispatcher
i weryfikator, a następnie dodaj własny negatywny przypadek. Zapytaj nie tylko „czy test
przechodzi?”, ale „jaką konkretną złą implementację ten test odrzuca?”.

Przy kolejnej sesji wykonaj Restart Kernel i świadomie wybierz tryb ponownie.
Usunięcie zmiennej z kluczem nie jest gwarancją wyczyszczenia pamięci procesu;
nie udostępniaj działającego kernela. Przed wysłaniem `.ipynb` wyczyść wszystkie wyniki.

Aktualna konfiguracja modelu: C02. Dostęp, przygotowanie i źródła: `02_Przygotowanie.md`.
Materiał sprawdzaj przed kolejnym wydarzeniem: modele i limity się zmieniają.
